# Fan-out join · **04 — Chart and table**

The chart that carries the argument: percentage of incomplete pathways
waiting over 18 weeks, by treatment function, computed twice. Once at
the verified pathway grain, once through the fan-out. The gap is labelled
on every row. The table underneath says why.

Everything here is read from notebook 03's outputs. No number is typed in,
including the ones in the alt text.

Colours: the correct-grain bar is blue and the fan-out bar orange (the
first two slots of the palette I use for every chart, checked for
colour-blind separation: protan ΔE 24.7). Both are labelled in the legend,
and the gap labels are text, so nothing depends on colour alone.

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import GOLD_DIR  # noqa: E402

IN = GOLD_DIR / "01_fan_out"
OUT = GOLD_DIR / "02_charts"
OUT.mkdir(parents=True, exist_ok=True)

cmp = pd.read_csv(IN / "fan_out_by_tf.csv")
# keep_default_na=False, or pandas reads the word "None" in my table as a missing value.
explain = pd.read_csv(IN / "explanatory_table.csv", keep_default_na=False)
contrast = pd.read_csv(IN / "contrast.csv")

SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1"
CORRECT, FANOUT = "#2a78d6", "#eb6834"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "text.color": INK,
                     "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK})

## The paired bar chart

Sorted by the signed gap, so the specialties the fan-out pushes up sit at
one end and those it pulls down at the other. Bars are thin, with a small
gap between the two bars of each pair.

In [2]:
d = cmp.sort_values("gap_points", ascending=True, ignore_index=True)   # largest positive gap at the top
names = d.tf_name.str.replace(r" Services?$", "", regex=True)
n = len(d)
bar_h, pair_gap = 0.34, 0.04

fig, ax = plt.subplots(figsize=(10, 11), dpi=200)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)

def hbar(y, width, colour):
    ax.barh(y, width, height=bar_h, color=colour, linewidth=0)

for i, r in d.iterrows():
    y_c = i + (bar_h / 2 + pair_gap / 2)
    y_f = i - (bar_h / 2 + pair_gap / 2)
    hbar(y_c, r.pct_over_18_correct, CORRECT)
    hbar(y_f, r.pct_over_18_fanout, FANOUT)
    ax.text(max(r.pct_over_18_correct, r.pct_over_18_fanout) + 0.8, i,
            f"{r.gap_points:+.2f} pts".replace("-", "−"), va="center", ha="left", fontsize=9.5, color=INK_2)

ax.set_yticks(range(n), names)
ax.set_ylim(-0.7, n - 0.3)
ax.set_xlim(0, 52)
ax.set_xticks(range(0, 51, 10), [f"{x}%" for x in range(0, 51, 10)])
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(axis="y", length=0); ax.tick_params(axis="x", length=0)
ax.set_xlabel("Incomplete pathways waiting over 18 weeks, August 2026")

# Legend, top-left, in the same order as each pair (correct above, fan-out below).
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=CORRECT, label="At pathway grain (matches NHS England Table 1)"),
                   Patch(color=FANOUT, label="Through a one-to-many ICB lookup (fan-out)")],
          loc="lower left", bbox_to_anchor=(0, 1.0), ncol=1, frameon=False, fontsize=10, handlelength=1.2)

infl = contrast.set_index("lookup").inflation.iloc[1]
fig.suptitle("Same file, same measure, one extra join. Neither bar looks wrong.",
             x=0.02, y=0.995, ha="left", fontsize=14, fontweight="bold")
fig.text(0.02, 0.968,
         f"% over 18 weeks by treatment function, England. The fan-out repeats each trust's rows once per ‘main’ ICB;\n"
         f"the denominator grows ×{infl:.2f} and no error is raised. Gap = fan-out minus correct, in percentage points.",
         ha="left", va="top", fontsize=10, color=INK_2)
fig.text(0.02, 0.008, "Source: NHS England, Consultant-led RTT Waiting Times, August 2026 (full CSV extract; press notice Table 1). "
         "Reported providers, NONC excluded.", fontsize=8, color=MUTED)
fig.tight_layout(rect=(0, 0.015, 1, 0.945))
fig.savefig(OUT / "paired_bars_fan_out.png", facecolor=SURFACE)
plt.show()

## The explanatory table, as an image for the article

In [3]:
fig, ax = plt.subplots(figsize=(11, 3.4), dpi=200)
fig.patch.set_facecolor(SURFACE); ax.axis("off")
cells = explain.values.tolist()
cols = ["", "At pathway grain", "Through a fan-out"]
tbl = ax.table(cellText=cells, colLabels=cols, cellLoc="left", colLoc="left", loc="upper left",
               colWidths=[0.30, 0.30, 0.40])
tbl.auto_set_font_size(False); tbl.set_fontsize(9.5); tbl.scale(1, 1.55)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor(GRID); cell.set_linewidth(0.6)
    cell.set_facecolor(SURFACE)
    cell.visible_edges = "B"
    if r == 0:
        cell.set_text_props(fontweight="bold", color=INK)
    elif c == 0:
        cell.set_text_props(color=INK_2)
fig.savefig(OUT / "explanatory_table.png", facecolor=SURFACE, bbox_inches="tight", pad_inches=0.15)
plt.show()

## Alt text, generated from the same numbers

In [4]:
top, bottom = d.iloc[-1], d.iloc[0]
eng = contrast.set_index("lookup")
alt = (
    f"Paired horizontal bar chart of the percentage of incomplete RTT pathways waiting over 18 weeks in England, "
    f"August 2026, for {n} treatment functions. Each function has two bars: one computed at the correct pathway grain, "
    f"which matches NHS England's published figures, and one computed through a join to a lookup that lists more than "
    f"one ICB for some trusts. The two bars in every pair are almost the same length. The largest gaps are "
    f"{top.tf_name.removesuffix(' Service')} at {top.gap_points:+.2f} percentage points "
    f"({top.pct_over_18_correct:.1f}% correct, {top.pct_over_18_fanout:.1f}% through the fan-out) and "
    f"{bottom.tf_name.removesuffix(' Service')} at {bottom.gap_points:+.2f} points "
    f"({bottom.pct_over_18_correct:.1f}% and {bottom.pct_over_18_fanout:.1f}%). "
    f"For England as a whole the fan-out inflates the pathway count from {int(eng.england_pathways.iloc[0]):,} to "
    f"{int(eng.england_pathways.iloc[1]):,} while the over-18-week rate moves only from "
    f"{eng.england_pct_over_18.iloc[0]:.1f}% to {eng.england_pct_over_18.iloc[1]:.1f}%. No error is raised."
).replace("+-", "−").replace("-0.", "−0.")
(OUT / "alt_text.md").write_text("# Alt text\n\n## paired_bars_fan_out.png\n\n" + alt + "\n")
print(alt)

Paired horizontal bar chart of the percentage of incomplete RTT pathways waiting over 18 weeks in England, August 2026, for 23 treatment functions. Each function has two bars: one computed at the correct pathway grain, which matches NHS England's published figures, and one computed through a join to a lookup that lists more than one ICB for some trusts. The two bars in every pair are almost the same length. The largest gaps are General Internal Medicine at +0.69 percentage points (25.8% correct, 26.5% through the fan-out) and Dermatology at −0.75 points (31.0% and 30.2%). For England as a whole the fan-out inflates the pathway count from 7,158,164 to 8,051,914 while the over-18-week rate moves only from 34.8% to 34.7%. No error is raised.
